# 05 — SHAP Explanations for Alerts & Importance Drift

**Project**: UPI Fraud Detection — Drift-Aware Explainable ML Framework
**Goal**: Per-alert SHAP explanations, global importance tracking, correlation with drift

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pathlib import Path
import joblib

from data_prep import load_ieee_cis, prepare_features_target
from models import evaluate_model
from drift_detection import DriftMonitor
from shap_explainer import SHAPExplainer, compute_shap_stability

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_ROOT = Path('../data')
MODELS_DIR = Path('../models')
REPORTS_DIR = Path('../reports')

print(f"Data root: {DATA_ROOT.absolute()}")

## 1. Load Data, Model & Initialize SHAP

In [ ]:
# Load data
ieee_df = load_ieee_cis(DATA_ROOT / 'ieee-cis')

def apply_fe(df, name):
    df = df.copy()
    if name == 'ieee-cis':
        df['hour'] = (df['TransactionDT'] // 3600) % 24
        df['day_of_week'] = (df['TransactionDT'] // 86400) % 7
        df['amount_log'] = np.log1p(df['TransactionAmt'])
        df['amount_round'] = (df['TransactionAmt'] == df['TransactionAmt'].round()).astype(int)
        if 'card1' in df.columns:
            df['card1_freq'] = df.groupby('card1')['card1'].transform('count')
    return df

ieee_df = apply_fe(ieee_df, 'ieee-cis')

def prepare_xy(df):
    drop_cols = ['TransactionID', 'TransactionDT', 'isFraud']
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df['isFraud'].astype('int8')
    for col in X.select_dtypes(include=['category']).columns:
        X[col] = X[col].cat.codes
    return X.fillna(-999), y

X_all, y_all = prepare_xy(ieee_df)
feature_names = X_all.columns.tolist()
print(f"Features: {len(feature_names)}")

In [ ]:
# Load model
model_path = MODELS_DIR / 'ieee_lightgbm.joblib'
if model_path.exists():
    model, _ = joblib.load(model_path), {}
    meta_path = model_path.with_suffix('.json')
    if meta_path.exists():
        import json
        with open(meta_path) as f:
            meta = json.load(f)
    print(f"Loaded: {meta.get('model', 'unknown')}")
else:
    from models import get_model
    from data_prep import handle_class_imbalance
    train_ieee = pd.read_parquet(DATA_ROOT / 'ieee-cis' / 'splits' / 'train.parquet')
    train_ieee = apply_fe(train_ieee, 'ieee-cis')
    X_train, y_train = prepare_xy(train_ieee)
    params = handle_class_imbalance(X_train, y_train)
    model = get_model('lightgbm', params)
    model.fit(X_train, y_train)
    print("Trained LightGBM on the fly")

# Get scores
ieee_df['risk_score'] = model.predict_proba(X_all)[:, 1]
ieee_df['day'] = (ieee_df['TransactionDT'] // 86400).astype(int)

In [ ]:
# Initialize SHAP Explainer
# Use training data as background
train_ieee = pd.read_parquet(DATA_ROOT / 'ieee-cis' / 'splits' / 'train.parquet')
train_ieee = apply_fe(train_ieee, 'ieee-cis')
X_train_bg, _ = prepare_xy(train_ieee)

explainer = SHAPExplainer(model, X_train_bg, feature_names=feature_names, max_background=500)
print("SHAP Explainer initialized")

## 2. Per-Alert SHAP Explanations

In [ ]:
# Get high-risk transactions (top 1% by score) from test period
test_data = ieee_df[ieee_df['day'] >= 60].copy()
X_test, y_test = prepare_xy(test_data)

# Top risky transactions
top_risky = test_data.nlargest(10, 'risk_score')
X_top = X_test.loc[top_risky.index]

print("=== Top 10 Riskiest Transactions ===")
for idx, row in top_risky.iterrows():
    x_row = X_top.loc[idx]
    explanation = explainer.explain_instance(x_row.to_frame().T, max_display=10)
    
    print(f"\nTransaction {idx}: Score={row['risk_score']:.4f}, True Label={row['isFraud']}")
    print(f"  Prediction: {explanation['prediction']:.4f}")
    print(f"  Top contributors:")
    for feat, val in explanation['feature_contributions'][:5]:
        print(f"    {feat}: {val:+.4f}")

In [ ]:
# Generate waterfall plot for first high-risk transaction
first_idx = top_risky.index[0]
x_first = X_top.loc[first_idx]
print(f"Generating waterfall for transaction {first_idx} (score={top_risky.loc[first_idx, 'risk_score']:.4f})")
explainer.plot_waterfall(x_first.to_frame().T, max_display=15, show=True)

## 3. Global Feature Importance (Mean |SHAP|)

In [ ]:
# Global importance on test set
importance_df = explainer.global_importance(X_test, max_samples=2000)
print("=== Top 20 Global Features (Mean |SHAP|) ===")
display(importance_df.head(20))  # type: ignore[unknown-name]

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
top20 = importance_df.head(20).iloc[::-1]
ax.barh(range(len(top20)), top20['mean_abs_shap'], xerr=top20['std_shap'], capsize=3)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['feature'])
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Global Feature Importance (Test Set)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP summary plot (beeswarm)
explainer.plot_summary(X_test.sample(n=min(2000, len(X_test)), random_state=42), plot_type='dot')

## 4. Importance Tracking Over Time Windows

In [ ]:
# Track importance drift across 7-day windows
max_day = ieee_df['day'].max()
window_size = 7
step_size = 7  # Non-overlapping windows

importance_records = []
for window_start in range(30, max_day - window_size + 1, step_size):
    window_end = window_start + window_size
    window_data = ieee_df[(ieee_df['day'] >= window_start) & (ieee_df['day'] < window_end)]
    
    if len(window_data) < 100:
        continue
    
    X_win, y_win = prepare_xy(window_data)
    
    # Global importance for this window
    imp_df = explainer.global_importance(X_win, max_samples=min(1000, len(X_win)))
    
    # Track drift
    drift_result = explainer.track_importance_drift(imp_df, timestamp=f"day_{window_start}_{window_end}")
    
    importance_records.append({
        'window_start': window_start,
        'window_end': window_end,
        'importance_df': imp_df,
        'drift': drift_result.get('drift_from_previous'),
    })
    
    if drift_result.get('drift_from_previous'):
        drift = drift_result['drift_from_previous']
        print(f"Window {window_start}-{window_end}: Spearman={drift['spearman_correlation']:.3f}, Top-20 Overlap={drift['top_k_overlap']:.2f}")

## 5. Importance Drift Visualization

In [ ]:
# Get importance history
history_df = explainer.get_importance_history()
print(f"Importance history shape: {history_df.shape}")

# Plot top features over time
if not history_df.empty:
    overall_top = history_df.groupby('feature')['importance'].mean().nlargest(10).index.tolist()
    
    fig, axes = plt.subplots(len(overall_top), 1, figsize=(12, 3*len(overall_top)), sharex=True)
    if len(overall_top) == 1:
        axes = [axes]
    
    for ax, feat in zip(axes, overall_top):
        feat_data = history_df[history_df['feature'] == feat]
        ax.plot(feat_data['timestamp'], feat_data['importance'], 'b-o', markersize=4)
        ax.set_ylabel('Mean |SHAP|')
        ax.set_title(f'Feature: {feat}')
        ax.grid(True, alpha=0.3)
    
    axes[-1].set_xlabel('Window')
    plt.tight_layout()
    plt.show()

## 6. SHAP Stability Analysis

In [ ]:
# Compute stability metrics
stability_df = compute_shap_stability(history_df, top_k=20)
print("=== SHAP Stability (Top 20 Features) ===")
display(stability_df)  # type: ignore[unknown-name]

# Features with high CV (unstable importance)
unstable = stability_df[stability_df['cv'] > 0.5].sort_values('cv', ascending=False)
if not unstable.empty:
    print("\nUnstable features (CV > 0.5):")
    display(unstable)  # type: ignore[unknown-name]


## 7. Correlate SHAP Drift with PSI Alerts

In [ ]:
# Load drift alerts from previous notebook
alerts_path = REPORTS_DIR / 'drift_alerts_log.csv'
if alerts_path.exists():
    alerts_df = pd.read_csv(alerts_path)
    print(f"Loaded {len(alerts_df)} drift alerts")
    
    # Get feature alerts
    feature_alerts = alerts_df[alerts_df['type'] == 'feature_drift']
    if not feature_alerts.empty:
        alerted_features = feature_alerts['column'].unique()
        print(f"Features with drift alerts: {list(alerted_features)}")
        
        # Check if these features also have SHAP importance drift
        for feat in alerted_features:
            if feat in history_df['feature'].values:
                feat_history = history_df[history_df['feature'] == feat].sort_values('timestamp')
                cv = feat_history['importance'].std() / feat_history['importance'].mean()
                print(f"  {feat}: SHAP CV={cv:.3f}, Mean importance={feat_history['importance'].mean():.4f}")
else:
    print("No drift alerts found. Run 04_drift_monitoring.ipynb first.")

## 8. Save Results

In [ ]:
# Save importance history
history_df.to_csv(REPORTS_DIR / 'shap_importance_history.csv', index=False)
stability_df.to_csv(REPORTS_DIR / 'shap_stability.csv', index=False)
importance_df.to_csv(REPORTS_DIR / 'shap_global_importance.csv', index=False)
print(f"Results saved to {REPORTS_DIR}")

## 9. Summary & Next Steps

In [ ]:
print("""
=== SHAP EXPLANATIONS COMPLETE ===

✅ Per-alert SHAP explanations (waterfall plots)
✅ Global feature importance (mean |SHAP|)
✅ SHAP summary plots (beeswarm)
✅ Importance tracking over time windows
✅ SHAP stability analysis (CV, trend)
✅ Correlation with PSI drift alerts
✅ Results saved to ../reports/

NEXT: Notebook 06_retraining_strategies.ipynb
- Compare Static vs Scheduled vs Drift-Triggered retraining
- Operational metrics: FP/100k, retrain frequency, model staleness
- Statistical significance testing
""")